# Image classification — train and evaluate

Training is one call; the interesting part is *reading what comes out*. This notebook trains, plots the learning curves, evaluates on the test split, and shows you how to interpret each number.

In [ ]:
# Run this cell first.
# On Colab it clones the kit and installs what's missing; locally it just finds the repo root.
import os, sys, subprocess
from pathlib import Path

if "google.colab" in sys.modules and not Path("cvkit").exists():
    subprocess.run(["git", "clone", "-q", "https://github.com/standungan/computer-vision-starter-kit"], check=True)
    os.chdir("computer-vision-starter-kit")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "scikit-learn", "gradio"], check=True)
while not Path("cvkit").exists() and Path.cwd() != Path.cwd().parent:
    os.chdir(Path.cwd().parent)          # notebooks live 3 folders below the repo root
sys.path.insert(0, os.getcwd())
print("working in", os.getcwd())

In [ ]:
from cvkit.config import load_config
from cvkit.synthetic import make_demo_data
from cvkit.utils import seed_everything

# quick.yaml = small synthetic demo, runs on a laptop CPU in minutes.
# Swap in default.yaml (bigger, pretrained) or your own config + data when you're ready.
CONFIG = "tasks/classification/configs/quick.yaml"
cfg = load_config(CONFIG)
seed_everything(cfg["seed"])

if not Path(cfg["data"]["root"]).exists():          # first run: create the demo dataset
    make_demo_data("classification", cfg)
print("task:", "classification", "| data:", cfg["data"]["root"], "| outputs:", cfg["output"]["dir"])

## The configuration
Everything tunable is in one YAML file. These are the sections that matter:

In [ ]:
import yaml

for section in ('data', 'model', 'train'):
    print(section + ':')
    for line in yaml.safe_dump(cfg[section], sort_keys=False).splitlines():
        print('  ' + line)

## What are we about to train?

In [ ]:
from cvkit.utils import count_parameters
from tasks.classification.data import build_loaders
from tasks.classification.model import build_model

classes = build_loaders(cfg)["classes"]
model = build_model(cfg, {"classes": classes})
p = count_parameters(model)
print(f"arch {cfg['model']['arch']} | pretrained {cfg['model']['pretrained']} | frozen backbone {cfg['model']['freeze_backbone']}")
print(f"{p['total']:,} parameters, {p['trainable']:,} trainable ({p['trainable_pct']:.0f}%) | {len(classes)} output classes")
del model

## Train
Watch the `val` numbers: they are what picks the best epoch.

In [ ]:
from tasks.classification.train import train

result = train(cfg)
print('\nbest epoch:', result['best_epoch'], '|', {k: round(v, 4) for k, v in result['best_metrics'].items()})

## Learning curves
**Gap widening** between train and val → overfitting (more augmentation, fewer epochs, smaller model). **Both flat and poor** → underfitting (train longer, raise the learning rate, unfreeze the backbone). **Val bouncing hard** → learning rate too high, or the val split is small. Remember the quick preset trains for only a few epochs, so expect curves still on their way up.

In [ ]:
from cvkit.viz import plot_history
from IPython.display import display

fig = plot_history(Path(cfg['output']['dir']) / 'history.csv')
display(fig)

## Evaluate on the test split
Val chose the checkpoint; the **test split** was untouched until now, so it is the honest number. One rule: look at test results, *don't tune on them* — the moment you tweak settings to chase the test score, it stops measuring generalisation.

In [ ]:
from tasks.classification.evaluate import evaluate

metrics = evaluate(cfg)

In [ ]:
from IPython.display import Image as IPImage, display

reports = Path(cfg["output"]["dir"]) / "reports"
for name in ['confusion_matrix.png', 'top_losses.png']:
    print(name)
    display(IPImage(filename=str(reports / name)))

## How to read these results
* **Accuracy** — share of test images labelled correctly. Quote it with the test-set size: on 66 images one mistake is 1.5 points.
* **Per-class precision / recall / F1** (in the printed report) — precision: of what the model *called* class X, how much was X; recall: of what *was* X, how much it found. A class with high precision but low recall is being under-predicted.
* **Macro-F1** — F1 averaged over classes equally; stays honest when classes are imbalanced, where accuracy flatters.
* **Confusion matrix** — find the off-diagonal cells: those are *which* classes get mixed up. Two classes that confuse each other often look alike to a human too, or are labelled inconsistently.
* **Top losses** — the error-analysis step most people skip. For each image ask: is the **label wrong** (fix the data)? is it **genuinely ambiguous** (the ceiling is the labels, not the model)? or is the model **missing something obvious** (more data, targeted augmentation)?

## Experiments worth running
Each one is a one-line change. Re-run the training cell with a different config and compare the best val numbers (change `cfg` like this, then re-run from *Train*):

```python
cfg['model']['arch'] = 'resnet34'     # or any key, same as --set model.arch=resnet34
```

1. `model.freeze_backbone = True` — how far does feature extraction alone get you?
2. `model.arch = 'resnet34'` or `'mobilenet_v3_small'` — does capacity help, or just cost time? how much does a deploy-friendly model give up?
3. `train.epochs` ×2 — is the curve still climbing at the end?
4. `data.augment.rotation = 0` and `data.augment.vflip = False` — what does augmentation buy you on this data?

Change **one thing at a time**, keep the seed, and write down what you changed — otherwise you can't tell what helped.

Next: **[03_predict_and_deploy](03_predict_and_deploy.ipynb)**.